In [ ]:
import pandas as pd
import networkx as nx
import community as community_louvain
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.cm as cm

# --- Load your data ---
df = pd.read_csv('data/MC3_comms_cleaned.csv')

# --- Build the undirected communication graph ---
G = nx.Graph()
for _, row in df.iterrows():
    G.add_edge(row['source'], row['target'])

# --- Run Louvain community detection ---
partition = community_louvain.best_partition(G)
communities = set(partition.values())

# --- Prepare subgraphs for each community ---
community_nodes = {c: [n for n in G.nodes() if partition[n] == c] for c in communities}
community_graphs = {c: G.subgraph(nodes) for c, nodes in community_nodes.items()}

# --- Create subplots ---
n_communities = len(communities)
cols = min(n_communities, 3)
rows = (n_communities + cols - 1) // cols

fig = make_subplots(rows=rows, cols=cols, subplot_titles=[f"Community {c}" for c in communities])

color_map = cm.get_cmap('tab20', n_communities)
for idx, c in enumerate(sorted(communities)):
    subG = community_graphs[c]
    # More spacing for each community
    pos = nx.spring_layout(subG, seed=42, k=1.5)
    node_x, node_y, node_text = [], [], []
    for node in subG.nodes():
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
        node_text.append(node)
    node_color = f'rgba({int(color_map(c)[0]*255)},{int(color_map(c)[1]*255)},{int(color_map(c)[2]*255)},0.8)'
    node_trace = go.Scatter(
        x=node_x, y=node_y,
        mode='markers+text',
        text=node_text,
        textposition='top center',
        marker=dict(
            color=node_color,
            size=20,
            line=dict(width=2, color='black')
        ),
        hoverinfo='text',
        showlegend=False
    )
    edge_x, edge_y = [], []
    for src, tgt in subG.edges():
        x0, y0 = pos[src]
        x1, y1 = pos[tgt]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]
    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        line=dict(width=1, color='#888'),
        hoverinfo='none',
        mode='lines',
        showlegend=False
    )
    row = idx // cols + 1
    col = idx % cols + 1
    fig.add_trace(edge_trace, row=row, col=col)
    fig.add_trace(node_trace, row=row, col=col)

fig.update_layout(
    title="Subgraphs of Each Detected Community",
    height=350 * rows,
    width=400 * cols,
    margin=dict(l=10, r=10, t=40, b=10)
)
for i in range(1, n_communities + 1):
    fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False, row=(i-1)//cols+1, col=(i-1)%cols+1)
    fig.update_yaxes(showgrid=False, zeroline=False, showticklabels=False, row=(i-1)//cols+1, col=(i-1)%cols+1)
fig.show()

/var/folders/zn/kgdwhjsj6fx6m4zbd82g2p1m0000gn/T/ipykernel_48983/3867029733.py:31: MatplotlibDeprecationWarning:

The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.

